# GAN Example — Generating Handwritten Digits (MNIST)

A minimal **Generative Adversarial Network (GAN)** built with TensorFlow / Keras.

A GAN trains two networks against each other:

- **Generator** — turns a random noise vector into a fake `28x28` image.
- **Discriminator** — tries to tell real MNIST images from the generator's fakes.

As training proceeds the generator gets better at fooling the discriminator, and the
fake images start to look like real handwritten digits.

> Tested with `tensorflow==2.1` (matches `requirements.txt`). Runs on CPU, but a GPU is much faster.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print('TensorFlow', tf.__version__)

## 1. Load and prepare the data

We only need the images (not the labels). Pixels are scaled to the range `[-1, 1]`
so they match the generator's `tanh` output.

In [ ]:
(train_images, _), (_, _) = tf.keras.datasets.mnist.load_data()

train_images = train_images.reshape(-1, 28, 28, 1).astype('float32')
train_images = (train_images - 127.5) / 127.5  # scale to [-1, 1]

BUFFER_SIZE = 60000
BATCH_SIZE = 256

train_dataset = (tf.data.Dataset
                 .from_tensor_slices(train_images)
                 .shuffle(BUFFER_SIZE)
                 .batch(BATCH_SIZE))

print('Training images:', train_images.shape)

## 2. The Generator

Takes a 100-dim noise vector and upsamples it to a `28x28x1` image using transposed
convolutions.

In [ ]:
NOISE_DIM = 100

def make_generator():
    model = tf.keras.Sequential([
        layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(NOISE_DIM,)),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Reshape((7, 7, 256)),

        layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same',
                               use_bias=False, activation='tanh'),
    ])
    return model

generator = make_generator()
generator.summary()

In [ ]:
# Sanity check: a random untrained generator just produces noise
noise = tf.random.normal([1, NOISE_DIM])
generated_image = generator(noise, training=False)
plt.imshow(generated_image[0, :, :, 0], cmap='gray')
plt.title('Untrained generator output')
plt.axis('off');

## 3. The Discriminator

A small CNN classifier. Output > 0 means "real", < 0 means "fake" (we use logits).

In [ ]:
def make_discriminator():
    model = tf.keras.Sequential([
        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[28, 28, 1]),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1),
    ])
    return model

discriminator = make_discriminator()
discriminator.summary()

## 4. Losses and optimizers

- The **discriminator** wants to output 1 for real images and 0 for fakes.
- The **generator** wants the discriminator to output 1 for its fakes.

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

## 5. The training step

One step: generate fakes, score real + fake batches, compute both losses, and apply
gradients to each network. `@tf.function` compiles it into a fast graph.

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gen_grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gen_grads, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    return gen_loss, disc_loss

In [ ]:
# A fixed noise seed so we can watch the SAME samples improve over time
num_examples = 16
seed = tf.random.normal([num_examples, NOISE_DIM])

def plot_samples(model, test_input, epoch=None):
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(4, 4))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(predictions[i, :, :, 0] * 0.5 + 0.5, cmap='gray')
        plt.axis('off')
    if epoch is not None:
        fig.suptitle(f'Epoch {epoch}')
    plt.show()

## 6. Train

`EPOCHS = 30` gives recognisable digits. Lower it to make this notebook quick to run;
raise it for sharper results.

In [ ]:
EPOCHS = 30

for epoch in range(1, EPOCHS + 1):
    g_loss = d_loss = 0.0
    for image_batch in train_dataset:
        g_loss, d_loss = train_step(image_batch)

    print(f'Epoch {epoch:3d}/{EPOCHS}  gen_loss={float(g_loss):.3f}  disc_loss={float(d_loss):.3f}')
    if epoch % 5 == 0 or epoch == 1:
        plot_samples(generator, seed, epoch)

## 7. Final generated digits

In [ ]:
plot_samples(generator, tf.random.normal([num_examples, NOISE_DIM]))

### Things to try

- Increase `EPOCHS` and watch the digits sharpen.
- Change `NOISE_DIM` or the layer sizes.
- Save the generator with `generator.save('gan_generator.h5')` and reload it to sample new digits without retraining.